# 09 — Algorytm Metropolisa i symulowane wyżarzanie

> Status nauki: ⬜ not-started *(uzupełnij samodzielnie)*

**Źródło:** Abundo, §1.8 „L'algoritmo di Metropolis e il Simulated Annealing", s. 35–37 wydruku (PDF 36–38).

**Wymagane podstawy:** [06 — rozkład stacjonarny, ergodyczność](./06-rozklad-stacjonarny-i-ergodycznosc.ipynb), [07 — bilans szczegółowy](./07-okresowosc-bistochastycznosc-odwracalnosc.ipynb), Teorema 1.2 (notatnik [08](./08-potegi-macierzy-przejscia-spektralnie.ipynb)).

## Czego się uczymy i po co?

Pierwsze **zastosowanie inżynierskie** teorii: jak użyć łańcucha Markowa do znalezienia **minimum globalnego** trudnej funkcji na ogromnej przestrzeni konfiguracji (np. problem komiwojażera). Pomysł: zbudować łańcuch, którego rozkład stacjonarny koncentruje się na minimach — i po prostu go symulować. To fundament metod MCMC używanych w statystyce i ML.

## 📘 Problem i rozkład Gibbsa

Dana jest skończona przestrzeń konfiguracji $\Omega$ (stany $\sigma$) i funkcja $f: \Omega \to \mathbb{R}$ („energia"). Szukamy $\sigma_{opt}$ z $f(\sigma_{opt}) = \min_\sigma f(\sigma)$; zbiór optymalnych konfiguracji to $\Omega_{opt}$.

Dla temperatury $T > 0$ definiujemy **rozkład Gibbsa** (wzór (1.49)):

$$\Pi_T(\sigma) = \frac{\exp\left(-\frac{f(\sigma)}{T}\right)}{\sum_{\sigma'\in\Omega}\exp\left(-\frac{f(\sigma')}{T}\right)}$$

(mianownik — stała normalizująca — gwarantuje $\sum_\sigma \Pi_T(\sigma) = 1$).

Zachowanie w granicach:
- $T$ **duże**: rozkład prawie jednostajny na $\Omega$ (wszystko porównywalnie prawdopodobne — faza eksploracji);
- $T$ **małe**: masa koncentruje się na konfiguracjach o małej energii $f$; dla $T \to 0^+$ — na rozkładzie jednostajnym na $\Omega_{opt}$.

Strategia: **próbkować z $\Pi_T$ dla małego $T$**. Ale jak próbkować z rozkładu, którego mianownika nie umiemy policzyć (suma po ogromnym $\Omega$)? Odpowiedź: łańcuchem Markowa.

## 📘 Konstrukcja łańcucha (wzór (1.50))

1. Dla każdego $\sigma$ wybieramy zbiór **sąsiadów** $\Omega_\sigma \subset \Omega$ (możliwe ruchy), symetrycznie ($\sigma \in \Omega_{\sigma'} \iff \sigma' \in \Omega_\sigma$) i o stałej liczności $\#\Omega_\sigma = \Theta$. Sąsiedztwo musi czynić łańcuch **nieprzywiedlnym i nieokresowym** — wtedy Teorema 1.2 gwarantuje istnienie rozkładu stacjonarnego.
2. Prawdopodobieństwa przejścia:

$$p_T(\sigma, \sigma') = \begin{cases} 0 & \sigma' \notin \Omega_\sigma \\ \mathrm{cost}\cdot \frac{1}{T}\exp\left\{-(f(\sigma') - f(\sigma))^+\right\} & \sigma' \in \Omega_\sigma \end{cases}$$

gdzie $g^+ = \max(g, 0)$ to **część dodatnia**: ruch „w dół" energii ($f(\sigma')\le f(\sigma)$) ma wykładnik 0, ruch „pod górę" jest karany czynnikiem $e^{-\Delta f/T}$.

**Kluczowy fakt:** $\Pi_T$ spełnia z tym łańcuchem **równanie bilansu szczegółowego** (1.51): $p_T(\sigma,\sigma')\,\Pi_T(\sigma) = p_T(\sigma',\sigma)\,\Pi_T(\sigma')$, więc (notatnik 07) $\Pi_T$ jest rozkładem niezmienniczym — i stacjonarnym, jedynym (ergodico II). Zauważ: do sprawdzenia bilansu **stała normalizująca się skraca** — nie musimy jej znać!

### W praktyce (schemat Metropolisa)

Będąc w $\sigma$: wylosuj sąsiada $\sigma'$ jednostajnie z $\Omega_\sigma$; zaakceptuj ruch z prawdopodobieństwem $\exp\{-(f(\sigma')-f(\sigma))^+/T\}$ (ruchy w dół — zawsze; pod górę — czasem), w przeciwnym razie zostań w $\sigma$. *(Uwaga redakcyjna: w (1.50) podręcznik umieszcza czynnik $\frac1T$ przed eksponentą i pisze prawdopodobieństwo akceptacji $p = \frac1T\exp\{\cdot\}$; w standardowym algorytmie Metropolisa akceptacja to po prostu $\exp\{-(\Delta f)^+/T\}$ — i taką wersję implementujemy niżej. Bilans szczegółowy zachodzi w obu wersjach, bo czynnik $\frac1T$ jest symetryczny w $\sigma,\sigma'$.)*

### 📘 Symulowane wyżarzanie (Simulated Annealing, SAA)

Zamiast stałego małego $T$ (ryzyko utknięcia w minimum lokalnym) **powoli schładzamy**: $T_n = c/\log n$. Geman i Geman udowodnili, że przy tak wolnym chłodzeniu rozkład graniczny to rozkład jednostajny na $\Omega_{opt}$ — algorytm znajduje minimum globalne. Nazwa: *wyżarzanie* jak w metalurgii (powolne studzenie metalu usuwa defekty).

Podręcznik wspomina zastosowania: problem komiwojażera ($\Omega$ = permutacje miast), *multiple digest problem* w biologii molekularnej, rekonstrukcja czasów zgłoszeń telefonicznych.

In [1]:
import numpy as np

# 🧪 Przykład własny: minimalizacja funkcji na Z_30 algorytmem Metropolisa
# f ma minimum globalne w x=7 i zwodnicze minimum lokalne w x=22
rng = np.random.default_rng(seed=5)
N_STANOW = 30
x = np.arange(N_STANOW)
f = 0.5*np.abs(x - 7) + 3*np.sin(x/2) + 5          # "energia"
f = f - f.min() + 1
x_opt = int(np.argmin(f))
print("minimum globalne w x =", x_opt)

def metropolis(T, kroki=20_000, start=25):
    sigma, wizyty = start, np.zeros(N_STANOW)
    for _ in range(kroki):
        sasiad = (sigma + rng.choice([-1, 1])) % N_STANOW   # sąsiedzi: +-1 (cyklicznie)
        delta = f[sasiad] - f[sigma]
        if delta <= 0 or rng.random() < np.exp(-delta / T): # akceptacja Metropolisa
            sigma = sasiad
        wizyty[sigma] += 1
    return wizyty / kroki

for T in [10.0, 1.0, 0.3]:
    czest = metropolis(T)
    print(f"T = {T:4}: łańcuch najczęściej w x = {int(np.argmax(czest))}, "
          f"częstość w x_opt = {czest[x_opt]:.3f}")

minimum globalne w x = 9


T = 10.0: łańcuch najczęściej w x = 8, częstość w x_opt = 0.066


T =  1.0: łańcuch najczęściej w x = 9, częstość w x_opt = 0.313


T =  0.3: łańcuch najczęściej w x = 21, częstość w x_opt = 0.000


In [2]:
# porównanie częstości wizyt z rozkładem Gibbsa (teoria vs symulacja), T = 1
T = 1.0
gibbs = np.exp(-f / T); gibbs /= gibbs.sum()
czest = metropolis(T, kroki=200_000)
print("stan | Gibbs  | częstość wizyt")
for i in range(5, 12):
    print(f"  {i:2d} | {gibbs[i]:.4f} | {czest[i]:.4f}")
# częstości wizyt ~ Pi_T: łańcuch rzeczywiście próbkuje z rozkładu Gibbsa

stan | Gibbs  | częstość wizyt
   5 | 0.0028 | 0.0029
   6 | 0.0185 | 0.0174
   7 | 0.1331 | 0.1333
   8 | 0.2729 | 0.2751
   9 | 0.3209 | 0.3206
  10 | 0.1841 | 0.1813
  11 | 0.0522 | 0.0508


In [3]:
# symulowane wyżarzanie: T_n = c / log(n) — powolne chłodzenie
rng = np.random.default_rng(seed=11)
sigma, c = 25, 8.0
for n in range(2, 60_000):
    T = c / np.log(n)
    sasiad = (sigma + rng.choice([-1, 1])) % N_STANOW
    delta = f[sasiad] - f[sigma]
    if delta <= 0 or rng.random() < np.exp(-delta / T):
        sigma = sasiad
print("po wyżarzaniu łańcuch zatrzymał się w x =", sigma, " (optimum:", x_opt, ")")

po wyżarzaniu łańcuch zatrzymał się w x = 8  (optimum: 9 )


## Typowe błędy

1. **Za szybkie chłodzenie** — teoria wymaga $T_n = c/\log n$ (bardzo wolno!); szybkie chłodzenie grozi utknięciem w minimum lokalnym.
2. Zapominanie o $(\cdot)^+$: ruchy obniżające energię akceptujemy **zawsze**; kara wykładnicza dotyczy tylko ruchów pod górę.
3. Źle dobrane sąsiedztwo może zepsuć nieprzywiedlność (nie każdy stan osiągalny) — wtedy zbieżność do $\Pi_T$ nie obowiązuje.
4. Mylenie rozkładu Gibbsa z „prawdopodobieństwem sukcesu" — $\Pi_T$ to rozkład **stacjonarny** łańcucha; potrzeba wielu kroków, by się do niego zbliżyć.

## Podsumowanie

- Rozkład Gibbsa $\Pi_T \propto e^{-f/T}$ koncentruje się na minimach $f$, gdy $T \to 0$.
- Łańcuch Metropolisa ma $\Pi_T$ jako rozkład stacjonarny — dzięki bilansowi szczegółowemu, bez znajomości stałej normalizującej.
- SAA = Metropolis + powolne chłodzenie $T_n = c/\log n$; zbiega do rozkładu jednostajnego na minimach globalnych.

## Pytania kontrolne

1. Dlaczego do działania algorytmu nie potrzebujemy mianownika rozkładu Gibbsa?
2. Co się dzieje z łańcuchem Metropolisa przy $T \to \infty$?
3. Energie: $f(\sigma) = 2$, $f(\sigma') = 5$, $T = 1$. Z jakim prawdopodobieństwem zaakceptujemy ruch $\sigma \to \sigma'$? A ruch $\sigma' \to \sigma$?

*(Odpowiedzi poniżej.)*

## Odpowiedzi

1. Bo warunek akceptacji zależy tylko od **różnicy** $f(\sigma') - f(\sigma)$; w bilansie szczegółowym stała normalizująca występuje po obu stronach i się skraca.
2. Prawdopodobieństwo akceptacji $e^{-\Delta f/T} \to 1$ — łańcuch akceptuje wszystko i błądzi jednostajnie po $\Omega$ (czysta eksploracja, rozkład graniczny jednostajny).
3. $\sigma\to\sigma'$ (pod górę o 3): $e^{-3/1} = e^{-3} \approx 0.0498$. $\sigma'\to\sigma$ (w dół): część dodatnia $= 0$, akceptacja $e^0 = 1$ (zawsze).

## Mini słowniczek IT–PL

| włoski | polski |
|---|---|
| ottimizzazione | optymalizacja |
| minimo/massimo globale | minimum/maksimum globalne |
| configurazione | konfiguracja |
| distribuzione di Gibbs | rozkład Gibbsa |
| temperatura | temperatura |
| raffreddamento (simulato) | (symulowane) chłodzenie/wyżarzanie |
| stati vicini | stany sąsiednie |
| parte positiva | część dodatnia |
| il problema del commesso viaggiatore | problem komiwojażera |
| costante di normalizzazione | stała normalizująca |